In [27]:
import sys
import importlib
from google.colab import drive
import pandas as pd
import numpy as np
# 1. Mount Drive
drive.mount('/content/drive')

# 2. Add path to your scripts
path_to_scripts = '/content/drive/My Drive/Colab Notebooks'
if path_to_scripts not in sys.path:
    sys.path.append(path_to_scripts)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
def process_interval_data(df, year=2025):
    df['Month'] = df['Month'].astype(str)
    df['Day'] = df['Day'].astype(int)

    month_to_num = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
        'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    if df['Month'].iloc[0] in month_to_num:
        df['Month'] = df['Month'].map(month_to_num)


    df['Date'] = pd.to_datetime(f"{year}-" + df['Month'].astype(str) + '-' + df['Day'].astype(str))
    df['Interval_TD'] = pd.to_timedelta(df['Interval'].astype(str))
    df['Full_Timestamp'] = df['Date'] + df['Interval_TD']
    df['Full_Timestamp'] = df['Full_Timestamp'].dt.tz_localize('UTC')
    df = df.drop(columns=['Date', 'Interval_TD'])

    start_ts = df['Full_Timestamp'].min()
    end_ts = df['Full_Timestamp'].max()
    full_range = pd.date_range(start=start_ts, end=end_ts, freq='30min', tz='UTC')

    master_df = pd.DataFrame({'Full_Timestamp': full_range})
    df_cleaned = pd.merge(master_df, df, on='Full_Timestamp', how='left')


    df_cleaned['Interval'] = df_cleaned['Full_Timestamp'].dt.strftime('%H:%M:%S')
    df_cleaned['Month'] = df_cleaned['Full_Timestamp'].dt.month
    df_cleaned['Day'] = df_cleaned['Full_Timestamp'].dt.day

    return df_cleaned

In [29]:
def process_datetime_daily(df):
  df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%y %a')
  return df

In [30]:
xls = pd.ExcelFile('/content/drive/MyDrive/Colab Notebooks/Data for Datathon (Revised).xlsx')
df_interval1 = pd.read_excel(xls, 'A - Interval')
df_interval2 = pd.read_excel(xls, 'B - Interval')
df_interval3 = pd.read_excel(xls, 'C - Interval')
df_interval4 = pd.read_excel(xls, 'D - Interval')
df_daily1 = pd.read_excel(xls, 'A - Daily')
df_daily2 = pd.read_excel(xls, 'B - Daily')
df_daily3 = pd.read_excel(xls, 'C - Daily')
df_daily4 = pd.read_excel(xls, 'D - Daily')

In [31]:
df_interval1 = process_interval_data(df_interval1)
df_interval2 = process_interval_data(df_interval2)
df_interval3 = process_interval_data(df_interval3)
df_interval4 = process_interval_data(df_interval4)
df_daily1 = process_datetime_daily(df_daily1)
df_daily2 = process_datetime_daily(df_daily2)
df_daily3 = process_datetime_daily(df_daily3)
df_daily4 = process_datetime_daily(df_daily4)



In [32]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
import os
import warnings
warnings.filterwarnings("ignore")

In [33]:
sheet_config = {
    "A - Daily": {"id_cols": ["Date"]},
    "B - Daily": {"id_cols": ["Date"]},
    "C - Daily": {"id_cols": ["Date"]},
    "D - Daily": {"id_cols": ["Date"]},
    "A - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "B - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "C - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "D - Interval": {"id_cols": ["Month", "Day", "Interval"]},
    "Daily Staffing": {"id_cols": ["Unnamed: 0"]},
}


In [34]:
def choose_arima_order(series, max_p=4, max_q=2, max_d=2):
    """
    Pick a simple ARIMA order using ADF test for d
    and AIC for p and q.
    """
    s = series.dropna()

    if len(s) < 20:
        return (1, 0, 0)

    d = 0
    temp = s.copy()

    for _ in range(max_d):
        try:
            p_value = adfuller(temp)[1]
            if p_value < 0.05:
                break
            temp = temp.diff().dropna()
            d += 1
        except:
            break

    best_order = (1, d, 1)
    best_aic = np.inf

    for p in range(max_p + 1):
        for q in range(max_q + 1):
            if p == 0 and q == 0:
                continue

            try:
                model = SARIMAX(
                    s,
                    order=(p, d, q),
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                result = model.fit(disp=False)

                if result.aic < best_aic:
                    best_aic = result.aic
                    best_order = (p, d, q)
            except:
                continue

    return best_order

In [35]:
def kalman_fill(series, order=None):
    """
    Fill missing values using SARIMAX + Kalman smoothing.
    If that fails, use interpolation instead.
    """
    if series.isna().sum() == 0:
        return series.copy()

    if order is None:
        order = choose_arima_order(series)

    try:
        model = SARIMAX(
            series,
            order=order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        result = model.fit(disp=False)

        smoothed_values = pd.Series(
            result.smoother_results.smoothed_forecasts[0],
            index=series.index
        )

        filled = series.copy()
        filled[series.isna()] = smoothed_values[series.isna()]
        return filled

    except Exception as e:
        print("Kalman smoothing failed, using interpolation instead:", e)
        return series.interpolate().bfill().ffill()


In [36]:
def impute_one_sheet(sheet_name, df, verbose=True):
    """
    Impute missing numeric columns in one sheet.
    """
    if sheet_name not in sheet_config:
        if verbose:
            print(f"{sheet_name} not found in config, returning original data.")
        return df.copy()

    id_cols = sheet_config[sheet_name]["id_cols"]
    new_df = df.copy()

    for col in df.columns:
        if col in id_cols:
            continue

        if not pd.api.types.is_numeric_dtype(df[col]):
            continue

        if df[col].isna().sum() == 0:
            continue

        missing_count = df[col].isna().sum()

        if verbose:
            print(f"[{sheet_name}] Working on column: {col}")
            print(f"Missing values: {missing_count}")

        order = choose_arima_order(df[col])

        if verbose:
            print(f"Chosen ARIMA order: {order}")

        filled_col = kalman_fill(df[col], order=order)

        observed_min = df[col].min()
        observed_max = df[col].max()
        filled_col = filled_col.clip(lower=observed_min, upper=observed_max)

        if filled_col.isna().sum() > 0:
            if verbose:
                print(f"Still missing values in {col}, using forward/backward fill.")
            filled_col = filled_col.ffill().bfill()

        new_df[col] = filled_col
    return new_df


In [37]:
def impute_all_sheets(datasets):
    final_data = {}
    print("I am Here!")
    for sheet_name, df in datasets.items():
        if sheet_name in sheet_config:
            final_data[sheet_name] = impute_one_sheet(sheet_name, df)
        else:
            final_data[sheet_name] = df.copy()

    # quick check for remaining NaNs
    print("\nChecking remaining missing values...")
    for sheet_name, df in final_data.items():
        total_missing = df.select_dtypes(include=[np.number]).isna().sum().sum()
        print(f"{sheet_name}: {total_missing} missing values")


    print("Saving CSV files to")

    for sheet_name, df in final_data.items():
        safe_name = sheet_name.replace(" ", "_").replace("-", "")+ ".csv"
        df.to_csv(safe_name, index=False)

    print("Done saving files.")

    return final_data


if __name__ == "__main__":
    print("Load your Excel file into a dictionary first, then run impute_all_sheets(datasets).")

Load your Excel file into a dictionary first, then run impute_all_sheets(datasets).


In [38]:
datasets = {
    "A - Interval": df_interval1,
    "B - Interval": df_interval2,
    "C - Interval": df_interval3,
    "D - Interval": df_interval4,
    "A - Daily": df_daily1,
    "B - Daily": df_daily2,
    "C - Daily": df_daily3,
    "D - Daily": df_daily4
}

In [39]:

imputed_results = impute_all_sheets(datasets)

I am Here!
[A - Interval] Working on column: Service Level
Missing values: 430
Chosen ARIMA order: (3, 0, 2)
[A - Interval] Working on column: Call Volume
Missing values: 373
Chosen ARIMA order: (4, 0, 2)
[A - Interval] Working on column: Abandoned Calls
Missing values: 391
Chosen ARIMA order: (4, 0, 1)
[A - Interval] Working on column: Abandoned Rate
Missing values: 449
Chosen ARIMA order: (2, 0, 1)
[A - Interval] Working on column: CCT
Missing values: 452
Chosen ARIMA order: (3, 0, 2)
[B - Interval] Working on column: Service Level
Missing values: 213
Chosen ARIMA order: (3, 0, 2)
[B - Interval] Working on column: Call Volume
Missing values: 183
Chosen ARIMA order: (4, 0, 2)
[B - Interval] Working on column: Abandoned Calls
Missing values: 184
Chosen ARIMA order: (3, 0, 2)
[B - Interval] Working on column: Abandoned Rate
Missing values: 207
Chosen ARIMA order: (3, 0, 2)
[B - Interval] Working on column: CCT
Missing values: 202
Chosen ARIMA order: (2, 0, 2)
[C - Interval] Working on c

In [40]:
imputed_results

{'A - Interval':                 Full_Timestamp  Month  Day  Interval  Service Level  \
 0    2025-04-01 00:00:00+00:00      4    1  00:00:00         1.0000   
 1    2025-04-01 00:30:00+00:00      4    1  00:30:00         1.0000   
 2    2025-04-01 01:00:00+00:00      4    1  01:00:00         1.0000   
 3    2025-04-01 01:30:00+00:00      4    1  01:30:00         1.0000   
 4    2025-04-01 02:00:00+00:00      4    1  02:00:00         1.0000   
 ...                        ...    ...  ...       ...            ...   
 4363 2025-06-30 21:30:00+00:00      6   30  21:30:00         0.9688   
 4364 2025-06-30 22:00:00+00:00      6   30  22:00:00         1.0000   
 4365 2025-06-30 22:30:00+00:00      6   30  22:30:00         0.8333   
 4366 2025-06-30 23:00:00+00:00      6   30  23:00:00         0.8000   
 4367 2025-06-30 23:30:00+00:00      6   30  23:30:00         1.0000   
 
       Call Volume  Abandoned Calls  Abandoned Rate     CCT  
 0             5.0              0.0          0.0000  137

In [42]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

df_int = imputed_results["A - Interval"].copy()
df_day = imputed_results["A - Daily"].copy()
df_int['Date'] = df_int['Full_Timestamp'].dt.date
df_day['Date'] = pd.to_datetime(df_day['Date']).dt.date

df_merged = pd.merge(df_int, df_day, on='Date', suffixes=('_int', '_day'))

df_merged['Hour'] = df_merged['Full_Timestamp'].dt.hour
df_merged['Minute'] = df_merged['Full_Timestamp'].dt.minute
df_merged['DayOfWeek'] = df_merged['Full_Timestamp'].dt.dayofweek

In [43]:
df_interval1

,Full_Timestamp,Month,Day,Interval,Service Level,Call Volume,Abandoned Calls,Abandoned Rate,CCT
0,2025-04-01 00:00:00+00:00,4,1,00:00:00,1.0000,5.0,0.0,0.0000,137.60
1,2025-04-01 00:30:00+00:00,4,1,00:30:00,1.0000,5.0,0.0,0.0000,263.40
2,2025-04-01 01:00:00+00:00,4,1,01:00:00,1.0000,4.0,0.0,0.0000,333.25
3,2025-04-01 01:30:00+00:00,4,1,01:30:00,1.0000,3.0,0.0,0.0000,170.00
4,2025-04-01 02:00:00+00:00,4,1,02:00:00,1.0000,1.0,0.0,0.0000,667.00
...,...,...,...,...,...,...,...,...,...
4363,2025-06-30 21:30:00+00:00,6,30,21:30:00,0.9688,34.0,2.0,0.0588,319.81
4364,2025-06-30 22:00:00+00:00,6,30,22:00:00,1.0000,23.0,0.0,0.0000,300.22
4365,2025-06-30 22:30:00+00:00,6,30,22:30:00,0.8333,13.0,1.0,0.0769,189.50
4366,2025-06-30 23:00:00+00:00,6,30,23:00:00,0.8000,10.0,0.0,0.0000,323.10


In [44]:
df_merged

,Full_Timestamp,Month,Day,Interval,Service Level_int,Call Volume_int,Abandoned Calls,Abandoned Rate,CCT_int,Date,Call Volume_day,CCT_day,Service Level_day,Abandon Rate,Hour,Minute,DayOfWeek
0,2025-04-01 00:00:00+00:00,4,1,00:00:00,1.0000,5.0,0.0,0.0000,137.60,2025-04-01,5249.0,304.60,0.8755,0.0235,0,0,1
1,2025-04-01 00:30:00+00:00,4,1,00:30:00,1.0000,5.0,0.0,0.0000,263.40,2025-04-01,5249.0,304.60,0.8755,0.0235,0,30,1
2,2025-04-01 01:00:00+00:00,4,1,01:00:00,1.0000,4.0,0.0,0.0000,333.25,2025-04-01,5249.0,304.60,0.8755,0.0235,1,0,1
3,2025-04-01 01:30:00+00:00,4,1,01:30:00,1.0000,3.0,0.0,0.0000,170.00,2025-04-01,5249.0,304.60,0.8755,0.0235,1,30,1
4,2025-04-01 02:00:00+00:00,4,1,02:00:00,1.0000,1.0,0.0,0.0000,667.00,2025-04-01,5249.0,304.60,0.8755,0.0235,2,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4363,2025-06-30 21:30:00+00:00,6,30,21:30:00,0.9688,34.0,2.0,0.0588,319.81,2025-06-30,4987.0,328.38,0.6614,0.0712,21,30,0
4364,2025-06-30 22:00:00+00:00,6,30,22:00:00,1.0000,23.0,0.0,0.0000,300.22,2025-06-30,4987.0,328.38,0.6614,0.0712,22,0,0
4365,2025-06-30 22:30:00+00:00,6,30,22:30:00,0.8333,13.0,1.0,0.0769,189.50,2025-06-30,4987.0,328.38,0.6614,0.0712,22,30,0
4366,2025-06-30 23:00:00+00:00,6,30,23:00:00,0.8000,10.0,0.0,0.0000,323.10,2025-06-30,4987.0,328.38,0.6614,0.0712,23,0,0


In [45]:
features = ['Month', 'Day', 'Hour', 'Minute', 'DayOfWeek']
target = 'CCT_int'

X = df_merged[features]
y = df_merged[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [46]:
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train)

preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))



In [48]:
import pandas as pd
import numpy as np


df_test = pd.read_csv('/content/template_forecast_v00.csv')
df_test


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,0:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,August,1,0:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,August,1,1:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,August,1,1:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,August,1,2:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1484,August,31,22:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,August,31,22:30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,August,31,23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Processing Channel A...
Processing Channel B...
Processing Channel C...
Processing Channel D...
All values filled! Download 'Final_August_XGBoost_Forecast.csv' from the sidebar.


In [49]:
def create_features(df, target_col):
  df = df.copy()

  df['dow'] = df['ds'].dt.dayofweek
  df['month'] = df['ds'].dt.month
  df['weekofyear'] = df['ds'].dt.isocalendar().week.astype(int)
  df['is_weekend'] = df['dow'].isin([5,6]).astype(int)
  df['lag1'] = df[target_col].shift(1)
  df['lag7'] = df[target_col].shift(7)
  df['lag14'] = df[target_col].shift(14)
  df['roll7'] = df[target_col].shift(1).rolling(7).mean()
  df['roll_std7'] = df[target_col].shift(1).rolling(7).std()

  return df

In [51]:
import pandas as pd
import xgboost as xgb
import numpy as np

df_august = pd.read_csv('/content/template_forecast_v00.csv')

df_august['Hour'] = pd.to_datetime(df_august['Interval'], format='%H:%M').dt.hour
df_august['Minute'] = pd.to_datetime(df_august['Interval'], format='%H:%M').dt.minute
df_august['Month_Num'] = 8
df_august['Date_temp'] = pd.to_datetime('2025-08-' + df_august['Day'].astype(str))
df_august['DayOfWeek'] = df_august['Date_temp'].dt.dayofweek

X_predict_base = df_august[['Month_Num', 'Day', 'Hour', 'Minute', 'DayOfWeek']].copy()
X_predict_base.columns = ['month', 'day', 'hour', 'minute', 'dow']

channels = ['A', 'B', 'C', 'D']
predict_tasks = {
    'Call Volume': 'Calls_Offered',
    'Abandoned Calls': 'Abandoned_Calls',
    'CCT': 'CCT'
}

for ch in channels:
    df_hist = imputed_results[f"{ch} - Interval"].copy()
    df_hist = df_hist.rename(columns={'Full_Timestamp': 'ds'})

    for hist_col, template_prefix in predict_tasks.items():
        df_featured = create_features(df_hist, hist_col)
        model_features = ['month', 'dow', 'is_weekend', 'lag1', 'lag7', 'roll7']

        df_train_ready = df_featured.dropna(subset=model_features + [hist_col])
        X_train = df_train_ready[model_features]
        y_train = df_train_ready[hist_col]

        model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100)
        model.fit(X_train, y_train)

        X_predict_task = X_predict_base.copy()
        X_predict_task['is_weekend'] = X_predict_task['dow'].isin([5,6]).astype(int)
        X_predict_task['lag1'] = df_hist[hist_col].iloc[-1]
        X_predict_task['lag7'] = df_hist[hist_col].iloc[-7] if len(df_hist) >= 7 else df_hist[hist_col].mean()
        X_predict_task['roll7'] = df_hist[hist_col].tail(7).mean()

        target_csv_col = f"{template_prefix}_{ch}"
        preds = model.predict(X_predict_task[model_features])
        df_august[target_csv_col] = np.clip(preds, 0, None).round(2)

    offered_col = f"Calls_Offered_{ch}"
    abandoned_col = f"Abandoned_Calls_{ch}"
    rate_col = f"Abandoned_Rate_{ch}"

    df_august[rate_col] = (df_august[abandoned_col] / df_august[offered_col]).fillna(0).round(4)


cols_to_drop = ['Hour', 'Minute', 'Month_Num', 'Date_temp', 'DayOfWeek']
df_final = df_august.drop(columns=cols_to_drop)
df_final.to_csv('Final_August_XGBoost_Forecast_Fixed_v2.csv', index=False)


--- Processing Channel A ---
--- Processing Channel B ---
--- Processing Channel C ---
--- Processing Channel D ---

Success! All channels (including Abandon Rates) are ready.
